In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import pandas as pd

import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# DGP

In [ ]:

np.random.seed(42)
# --- Data-Generating Processes (DGPs) ---
def generate_data(scenario, T):
    y = np.zeros(T + 1)
    y[0] = np.random.randn()
    if scenario == 'A':
        for t in range(1, T + 1): y[t] = 0.5 * y[t-1] + np.random.randn()
    elif scenario == 'B':
        for t in range(1, T + 1):
            sigma_t = np.min([np.exp(0.5 * y[t-1])**0.5, 10])
            y[t] = 0.5 * y[t-1] + sigma_t * np.random.randn()
    elif scenario == 'C':
        for t in range(1, T + 1):
            phi = 0.8 if t <= (T // 2) else -0.8
            y[t] = phi * y[t-1] + np.random.randn()
    X = y[:-1].reshape(-1, 1)
    Y = y[1:]
    return X, Y, y

# --- Predictive Model and Score ---
def get_fixed_model(X_train, Y_train):
    model = LinearRegression()
    model.fit(X_train, Y_train)
    return model

def get_scores(model, X, Y):
    preds = model.predict(X)
    return np.abs(Y - preds)

# Simulation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from contextlib import contextmanager

import cp_methods as cpm

In [ ]:
def build_sim_data(
    scenario,
    T=1500,
    BURN_IN=500,
    R=200,
    spci_w_lag=8,
    spci_T_train=None,
):
    """
    Build one-series cache compatible with cp_methods.eval_*.
    """
    if spci_T_train is None:
        spci_T_train = R

    X, Y, y_full = generate_data(scenario, T)

    # fixed predictor and scores
    fixed_model = get_fixed_model(X[:BURN_IN], Y[:BURN_IN])
    pred = np.asarray(fixed_model.predict(X), dtype=float).ravel()
    score = np.asarray(get_scores(fixed_model, X, Y), dtype=float).ravel()

    # warmup so imported eval_* can use the same recent history as handwritten sim
    warmup = max(R, spci_w_lag, spci_T_train)
    test_start = max(0, BURN_IN - warmup)

    data = [{
        "series_id": 0,
        "ts": pd.RangeIndex(T),
        "y": np.asarray(Y, dtype=float).ravel(),
        "pred": pred,
        "score": score,
        "X": np.asarray(X, dtype=float).reshape(-1, 1),  # scalar X for the sim
        "is_test": (np.arange(T) >= test_start),
        "const_test": False,
    }]

    meta = {
        "scenario": scenario,
        "T": T,
        "BURN_IN": BURN_IN,
        "R": R,
        "X_full": np.asarray(X, dtype=float).reshape(-1, 1),
        "Y_full": np.asarray(Y, dtype=float).ravel(),
        "pred_full": pred,
        "score_full": score,
        "fixed_model": fixed_model,
    }
    return data, meta

In [ ]:
SIM_SPCI_W_LAG = 8
SIM_SPCI_T_TRAIN = 200
SIM_SPCI_REFIT = 100   # set 1 for most faithful SPCI

SIM_METHOD_RUNNERS = {
    "cp": cpm.eval_cp,
    "lcp": cpm.eval_lcp,
    "aci": cpm.eval_aci,
    "dtaci": cpm.eval_dtaci,
    "olcp": cpm.eval_olcp,
    "olcph": cpm.eval_olcp_adahedge,
    "spci": lambda data, alpha, R, exclude_const_test=False: cpm.eval_spci(
        data,
        alpha=alpha,
        R=R,
        w_lag=SIM_SPCI_W_LAG,
        T_train=SIM_SPCI_T_TRAIN,
        refit_every=SIM_SPCI_REFIT,
        exclude_const_test=exclude_const_test,
        show_pbar=False,
    ),
}

In [ ]:
T=1500
BURN_IN=500
R=200
ALPHA=0.1

def run_simulation_via_cp_methods(
    scenario,
    T=1500,
    BURN_IN=500,
    R=200,
    ALPHA=0.1,
    verbose=True,
):
    data, meta = build_sim_data(
        scenario=scenario,
        T=T,
        BURN_IN=BURN_IN,
        R=R,
        spci_w_lag=SIM_SPCI_W_LAG,
        spci_T_train=SIM_SPCI_T_TRAIN,
    )

    if verbose:
        print(f"\n--- Running Simulation for Scenario {scenario} ---")

    results = {}
    summary_rows = []
    for name, fn in SIM_METHOD_RUNNERS.items():
        df_m, sec = fn(data, alpha=ALPHA, R=R, exclude_const_test=False)

        # keep only the rows corresponding to the original evaluation window t >= BURN_IN
        df_m = df_m.loc[df_m["t"] >= BURN_IN].reset_index(drop=True)

        results[name] = df_m
        summary_rows.append({
            "Method": name.upper(),
            "Coverage": float(df_m["covered"].mean()),
            "AvgWidth": float(df_m["width"].replace([np.inf, -np.inf], np.nan).mean()),
            "Rows": int(len(df_m)),
            "Seconds": float(sec),
        })

    summary = pd.DataFrame(summary_rows).sort_values("Method").reset_index(drop=True)
    return results, meta, summary

In [ ]:
def merge_method_results(results, meta):
    burn_in = meta["BURN_IN"]
    T = meta["T"]

    base = pd.DataFrame({
        "time": np.arange(T),
        "true_y": meta["Y_full"],
        "pred_y": meta["pred_full"],
        "X": meta["X_full"].ravel(),
    })
    base = base.loc[base["time"] >= burn_in].reset_index(drop=True)

    for name, df_m in results.items():
        tmp = df_m[["t", "lower", "upper"]].copy()
        tmp = tmp.rename(columns={
            "t": "time",
            "lower": f"{name}_lower",
            "upper": f"{name}_upper",
        })
        base = base.merge(tmp, on="time", how="left")

    return base

In [ ]:
from pathlib import Path
import pickle

outdir = Path("sim_results_full")
outdir.mkdir(exist_ok=True)

saved_A = []
saved_B = []
saved_C = []

for rep in tqdm(range(1), desc="Round"):
    res_A, meta_A, summary_A = run_simulation_via_cp_methods(
        "A", T=T, BURN_IN=BURN_IN, R=R, ALPHA=ALPHA, verbose=False
    )
    obj_A = {"scenario": "A", "rep": rep, "res": res_A, "meta": meta_A, "summary": summary_A}
    saved_A.append(obj_A)
    with open(outdir / f"scenario_A_rep_{rep:03d}.pkl", "wb") as f:
        pickle.dump(obj_A, f)

    res_B, meta_B, summary_B = run_simulation_via_cp_methods(
        "B", T=T, BURN_IN=BURN_IN, R=R, ALPHA=ALPHA, verbose=False
    )
    obj_B = {"scenario": "B", "rep": rep, "res": res_B, "meta": meta_B, "summary": summary_B}
    saved_B.append(obj_B)
    with open(outdir / f"scenario_B_rep_{rep:03d}.pkl", "wb") as f:
        pickle.dump(obj_B, f)

    res_C, meta_C, summary_C = run_simulation_via_cp_methods(
        "C", T=T, BURN_IN=BURN_IN, R=R, ALPHA=ALPHA, verbose=False
    )
    obj_C = {"scenario": "C", "rep": rep, "res": res_C, "meta": meta_C, "summary": summary_C}
    saved_C.append(obj_C)
    with open(outdir / f"scenario_C_rep_{rep:03d}.pkl", "wb") as f:
        pickle.dump(obj_C, f)

In [ ]:
import pandas as pd
import numpy as np

def summarize_saved_runs(saved_runs, scenario_name):
    tabs = []
    for obj in saved_runs:
        df = obj["summary"].copy()
        df["rep"] = obj["rep"]
        df["scenario"] = scenario_name
        tabs.append(df)
    return pd.concat(tabs, ignore_index=True)

summary_A_all = summarize_saved_runs(saved_A, "A")
summary_B_all = summarize_saved_runs(saved_B, "B")
summary_C_all = summarize_saved_runs(saved_C, "C")

summary_all = pd.concat(
    [summary_A_all, summary_B_all, summary_C_all],
    ignore_index=True
)

method_order = ["CP", "LCP", "ACI", "DTACI", "SPCI", "OLCP", "OLCPH"]
summary_all["Method"] = pd.Categorical(
    summary_all["Method"],
    categories=method_order,
    ordered=True
)

agg = (
    summary_all
    .groupby(["scenario", "Method"], as_index=False, observed=True)
    .agg(
        Coverage_mean=("Coverage", "mean"),
        Coverage_std=("Coverage", "std"),
        AvgWidth_mean=("AvgWidth", "mean"),
        AvgWidth_std=("AvgWidth", "std"),
        Seconds_mean=("Seconds", "mean"),
        Seconds_std=("Seconds", "std"),
        Reps=("rep", "nunique"),
    )
    .sort_values(["scenario", "Method"])
    .reset_index(drop=True)
)

print(agg.to_string(index=False, formatters={
    "Coverage_mean": "{:.3f}".format,
    "Coverage_std": "{:.3f}".format,
    "AvgWidth_mean": "{:.3f}".format,
    "AvgWidth_std": "{:.3f}".format,
    "Seconds_mean": "{:.2f}".format,
    "Seconds_std": "{:.2f}".format,
}))